imports

In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf
from IPython.display import Audio
import matplotlib.pyplot as plt
import librosa.display


librispeech

In [ ]:
SPEAKER = "116"
CHAPTER = "288045"

AUDIO_DIR = f"/net/db/LibriSpeech/dev-other/{SPEAKER}/{CHAPTER}"

# wie viele Sätze vergleichen?
N = 4

# Dateien auswählen
files = sorted(
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.endswith(".flac") or f.endswith(".wav")
)

print("Gefundene Dateien:")
for i,f in enumerate(files):
    print(f"{i+1}: {os.path.basename(f)}")

selected_files = files[:N]

print("\nAusgewählte Dateien:")
for f in selected_files:
    print(os.path.basename(f))

# Audio laden
audio_list = []
sr_ref = None

for f in selected_files:
    audio, sr = librosa.load(f, sr=None)
    if sr_ref is None:
        sr_ref = sr
    else:
        assert sr == sr_ref
    audio_list.append(audio)

# concatenieren
combined = np.concatenate(audio_list)

print(f"\nLänge der {N} Sätze: {combined.shape[0]/sr_ref:.2f} Sekunden")

Audio(combined, rate=sr_ref)


download

In [ ]:
from IPython.display import HTML
import base64
from io import BytesIO

def download_button(fig, filename="plot.png"):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=300, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    href = (
        f'<a download="{filename}" href="data:image/png;base64,{b64}" '
        f'target="_blank">📥 Download Plot</a>'
    )
    return HTML(href)


plots

In [ ]:
# Waveform
def plot_waveform_with_boundaries_dl(combined, audio_list, sr, filename="waveform.png"):
    boundaries = np.cumsum([len(a) for a in audio_list])

    fig, ax = plt.subplots(figsize=(14,4))
    ax.plot(combined, linewidth=0.6)

    for b in boundaries:
        ax.axvline(b, color='red', linestyle='--', linewidth=1, alpha=0.7)

    ax.set_title("Waveform mit Satzgrenzen")
    ax.set_xlabel("Samples")
    ax.set_ylabel("Amplitude")

    plt.show()
    return download_button(fig, filename)


# RMS Energy 
def plot_energy_with_boundaries_dl(combined, audio_list, sr, filename="energy.png"):
    boundaries = np.cumsum([len(a) for a in audio_list])
    rms = librosa.feature.rms(y=combined)[0]
    frames = np.linspace(0, len(combined), num=len(rms))

    fig, ax = plt.subplots(figsize=(14,3))
    ax.plot(rms, linewidth=0.8)

    for b in boundaries:
        frame_idx = np.argmin(np.abs(frames - b))
        ax.axvline(frame_idx, color='red', linestyle='--', linewidth=1, alpha=0.7)

    ax.set_title("RMS Energy mit Satzgrenzen")
    ax.set_xlabel("Frame Index")
    ax.set_ylabel("Energy")

    plt.show()
    return download_button(fig, filename)


#Spectrogram 
def plot_spectrogram_with_boundaries_dl(combined, audio_list, sr, filename="spectrogram.png"):
    boundaries = np.cumsum([len(a) for a in audio_list])
    time_boundaries = boundaries / sr

    S = librosa.amplitude_to_db(np.abs(librosa.stft(combined)), ref=np.max)

    fig, ax = plt.subplots(figsize=(14,5))
    img = librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="hz", cmap="magma", ax=ax)

    for t in time_boundaries:
        ax.axvline(t, color='white', linestyle='--', linewidth=1, alpha=0.8)

    ax.set_title("Spektrogramm mit Satzgrenzen")
    fig.colorbar(img, ax=ax)

    plt.show()
    return download_button(fig, filename)


In [ ]:
plot_waveform_with_boundaries_dl(combined, audio_list[:N], sr_ref, f"waveform_N{N}.png")
plot_energy_with_boundaries_dl(combined, audio_list[:N], sr_ref, f"energy_N{N}.png")
plot_spectrogram_with_boundaries_dl(combined, audio_list[:N], sr_ref, f"spectrogram_N{N}.png")


FRAME MOS

In [ ]:
from local_sqa.modules.ssl_mos import SpeechQualityPredictor

MODEL_DIR = "/net/vol/zigor/checkpoints/19"
CHECKPOINT_NAME = "ckpt_best_SRCC.pth"

predictor = SpeechQualityPredictor(
    storage_dir=MODEL_DIR,
    checkpoint_name=CHECKPOINT_NAME,
    return_numpy=True,
    device="cpu"
)

wav_batch = combined[None, :]
num_samples = [combined.shape[0]]

global_mos, frame_mos, _ = predictor(wav_batch, num_samples)

frame_mos = frame_mos[0]
global_mos = float(global_mos[0])

print("Normal MOS:", global_mos)
print("Frame MOS shape:", frame_mos.shape)


plot

In [ ]:
def plot_frame_mos_with_boundaries_dl(frame_mos, audio_list, combined_len, filename="frame_mos.png"):
    boundaries = np.cumsum([len(a) for a in audio_list])
    frames = np.linspace(0, combined_len, num=len(frame_mos))

    fig, ax = plt.subplots(figsize=(16,4))
    ax.plot(frame_mos, linewidth=0.8)

    for b in boundaries:
        frame_idx = np.argmin(np.abs(frames - b))
        ax.axvline(frame_idx, color='red', linestyle='--', alpha=0.7)

    ax.set_title("Frame MOS (Normal)")
    ax.set_xlabel("Frame Index")
    ax.set_ylabel("MOS")

    plt.show()
    return download_button(fig, filename)

plot_frame_mos_with_boundaries_dl(
    frame_mos,
    audio_list[:N],
    combined.shape[0],
    "frame_mos_normal.png"
)


long

In [ ]:
# Long Ordner für dieses Kapitel
LONG_DIR = f"/net/db/librispeech_long/dev-other/{SPEAKER}/{CHAPTER}"

long_files = sorted(
    os.path.join(LONG_DIR, f)
    for f in os.listdir(LONG_DIR)
    if f.endswith(".flac")
)

print("Gefundene Long-Form Chunks:")
for f in long_files:
    print(os.path.basename(f))


long_segments = []
for f in long_files:
    audio, sr_long = sf.read(f)
    long_segments.append(audio)


long_audio = np.concatenate(long_segments)


assert sr_long == sr_ref


T = combined.shape[0]
combined_long = long_audio[:T]

print(f"Long-Form Gesamtlaenge: {len(long_audio)/sr_ref:.2f} s")
print(f"Vergleichs-Laenge (T): {T/sr_ref:.2f} s")

Audio(combined_long, rate=sr_ref)


FRAME MOS long

In [ ]:
wav_batch_long = combined_long[None, :]
global_mos_long, frame_mos_long, _ = predictor(wav_batch_long, [T])

frame_mos_long = frame_mos_long[0]
global_mos_long = float(global_mos_long[0])

print("Long MOS:", global_mos_long)
print("Frame MOS shape:", frame_mos_long.shape)


vergleich

In [ ]:
def plot_compare_frame_mos(
    frame_normal,
    frame_long,
    audio_list,
    combined_len,
    filename="compare_frame_mos.png"
):
    boundaries = np.cumsum([len(a) for a in audio_list])
    frames = np.linspace(0, combined_len, num=len(frame_normal))

    fig, ax = plt.subplots(figsize=(16,4))

    ax.plot(frame_normal, label="Normal", alpha=0.9)
    ax.plot(frame_long, label="Long-Form", alpha=0.9)

    for b in boundaries:
        frame_idx = np.argmin(np.abs(frames - b))
        ax.axvline(frame_idx, color='red', linestyle='--', alpha=0.5)

    ax.set_title("Frame MOS Vergleich: Normal vs Long-Form")
    ax.set_xlabel("Frame Index")
    ax.set_ylabel("MOS")
    ax.legend()
    ax.grid(alpha=0.2)

    plt.show()
    return download_button(fig, filename)

plot_compare_frame_mos(
    frame_mos,
    frame_mos_long,
    audio_list[:N],
    combined.shape[0],
    "compare_frame_mos.png"
)


wave vergleich

In [ ]:
# ffalls die beiden nicht exakt gleich lang sind; trimmen
L = min(len(combined), len(combined_long))
a = combined[:L]
b = combined_long[:L]

plt.figure(figsize=(18,4))
plt.plot(a, label="Normal", alpha=0.8)
plt.plot(b, label="Long-Form", alpha=0.8)

plt.title("Waveform Vergleich — Normal vs Long-Form")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import HTML
import base64
from io import BytesIO

def download_button(fig, filename="plot.png"):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=300, bbox_inches="tight")
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    href = (
        f'<a download="{filename}" href="data:image/png;base64,{b64}" '
        f'target="_blank">📥 Download Plot</a>'
    )
    return HTML(href)


BVCC TESTS

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from IPython.display import Audio, display


# config 
BVCC_TEST_LIST = Path("/net/db/BVCC/main/DATA/sets/test_mos_list.txt")
BVCC_WAV_ROOT  = Path("/net/db/BVCC/main/DATA/wav")

SYSTEM_ID = "sys753b8"   # <- hier ändern
TARGET_SECONDS = 180.0
START_INDEX = 0
MAX_FILES = 5000

# playback config
PLAY_EACH_SEGMENT = True
PLAY_COMBINED_PREVIEW = True
COMBINED_PREVIEW_SECONDS = 20.0   # nur preview, nicht ganze 180s
SR = None  



# helpers 
def parse_bvcc_list(list_path: Path):
    items = []
    for ln in list_path.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        fn, mos = ln.split(",")
        items.append((fn.strip(), float(mos)))
    return items

def pick_system_sequence(items, system_id: str, target_seconds: float, start_index: int = 0, max_files: int = 5000):
    filtered = [(fn, mos) for (fn, mos) in items if fn.startswith(f"{system_id}-")]
    if not filtered:
        raise ValueError(f"no items found for SYSTEM_ID={system_id}")

    start = start_index % len(filtered)
    ordered = filtered[start:] + filtered[:start]

    seq = []
    total_s = 0.0

    for fn, mos_t in ordered:
        if total_s >= target_seconds or len(seq) >= max_files:
            break

        p = BVCC_WAV_ROOT / fn


        try:
            dur = float(librosa.get_duration(path=str(p)))
        except Exception:
            # fallback: laden, falls get_duration streikt
            y, sr = librosa.load(str(p), sr=None, mono=True)
            dur = float(len(y) / sr)

        seq.append((fn, float(mos_t), p, dur))
        total_s += dur

    return seq, total_s, len(filtered)

def load_audio(path: Path, sr=None):
    y, s = librosa.load(str(path), sr=sr, mono=True)
    y = y.astype(np.float32)
    return y, s



# main: pick files
items = parse_bvcc_list(BVCC_TEST_LIST)
picked, total_s, sys_count = pick_system_sequence(
    items,
    system_id=SYSTEM_ID,
    target_seconds=TARGET_SECONDS,
    start_index=START_INDEX,
    max_files=MAX_FILES
)

print(f"SYSTEM_ID={SYSTEM_ID} -> available files in test list: {sys_count}")
print(f"picked files: {len(picked)}")
print(f"picked duration: {total_s:.2f}s (target={TARGET_SECONDS:.2f}s)")

df = pd.DataFrame(picked, columns=["filename", "mos_target", "path", "duration_s"])
display(df[["filename", "mos_target", "duration_s"]].head(50))


# playback function
def play_sequence(df_seq: pd.DataFrame, title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)

    # segments
    if PLAY_EACH_SEGMENT:
        for i, row in df_seq.reset_index(drop=True).iterrows():
            fn = row["filename"]
            mos_t = row["mos_target"]
            dur = row["duration_s"]
            p = row["path"]

            print(f"\n[{i+1:02d}] {fn} | target MOS={mos_t:.3f} | dur={dur:.2f}s")
            y, sr = load_audio(p, sr=SR)
            display(Audio(y, rate=sr))

    # combined preview (first N seconds)
    if PLAY_COMBINED_PREVIEW:
        y_all = []
        sr_ref = None

        for row in df_seq.itertuples(index=False):
            y, sr = load_audio(row.path, sr=SR)
            if sr_ref is None:
                sr_ref = sr
            elif sr != sr_ref:
                raise RuntimeError(f"sample rate mismatch: got {sr} vs {sr_ref} for {row.path}")
            y_all.append(y)

        combined = np.concatenate(y_all) if len(y_all) else np.zeros(0, dtype=np.float32)

        preview_len = int(COMBINED_PREVIEW_SECONDS * sr_ref)
        preview = combined[:preview_len]

        print(f"\nCombined preview: {min(COMBINED_PREVIEW_SECONDS, len(combined)/sr_ref):.2f}s of {len(combined)/sr_ref:.2f}s total")
        display(Audio(preview, rate=sr_ref))


# play normal + reversed
play_sequence(df, f"BVCC {SYSTEM_ID} — NORMAL order (picked)")

df_rev = df.iloc[::-1].reset_index(drop=True)
play_sequence(df_rev, f"BVCC {SYSTEM_ID} — REVERSED order (picked[::-1])")


SOMOS

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from IPython.display import Audio, display


# config (SOMOS)
SOMOS_AUDIO_ROOT = Path("/net/db/somos/audios")

# choose ONE:
MOS_LIST = Path("/net/db/somos/training_files/split1/clean/test_mos_list.txt")
# MOS_LIST = Path("/net/db/somos/training_files/split1/full/test_mos_list.txt")

SYSTEM_IDS = ["061", "057", "110", "124", "191"]   # our 5 picks

TARGET_SECONDS = 180.0
START_INDEX = 0
MAX_FILES = 5000

# playback config
PLAY_EACH_SEGMENT = True
PLAY_COMBINED_PREVIEW = True
COMBINED_PREVIEW_SECONDS = 20.0
SR = None  # keep original sr

# if True, we try to infer systemId from utteranceId patterns:
# - utteranceId can look like: "<sentenceId>_<systemId>.wav" (often numeric systemId)
# - but some are like "LJ002-0181_110.wav" (systemId=110)
# if SOMOS format differs, you'll see "unknown" and we adjust.
STRICT_SYSTEM_PARSE = True



# helpers
def parse_somos_mos_list(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # expected: utteranceId, mean
    if "utteranceId" not in df.columns or "mean" not in df.columns:
        raise KeyError(f"unexpected columns in {path}: {list(df.columns)}")
    return df

def extract_system_id(utterance_id: str) -> str:
    # expects "..._<systemId>.wav"
    # example: LJ002-0181_110.wav -> "110"
    base = utterance_id
    if base.endswith(".wav"):
        base = base[:-4]
    if "_" not in base:
        return "unknown"
    tail = base.split("_")[-1]
    # systemId in SOMOS is 000-200, usually 3 digits
    if tail.isdigit():
        return tail.zfill(3)
    return "unknown"

def pick_system_sequence(df_all: pd.DataFrame, system_id: str, target_seconds: float, start_index: int, max_files: int):
    df = df_all.copy()
    df["system_id"] = df["utteranceId"].apply(extract_system_id)

    if STRICT_SYSTEM_PARSE and (df["system_id"] == "unknown").any():
        # still proceed, but warn loudly
        unk = int((df["system_id"] == "unknown").sum())
        print(f"[WARN] {unk} utteranceIds had unknown system_id parsing. If results look wrong, we must adjust parser.")

    df_sys = df[df["system_id"] == system_id.zfill(3)].reset_index(drop=True)
    if df_sys.empty:
        raise ValueError(f"no entries found for system_id={system_id} in {MOS_LIST}")

    # stable order + start offset (same behavior as BVCC script)
    start = start_index % len(df_sys)
    df_ord = pd.concat([df_sys.iloc[start:], df_sys.iloc[:start]], ignore_index=True)

    picked = []
    total_s = 0.0

    for _, row in df_ord.iterrows():
        if total_s >= target_seconds or len(picked) >= max_files:
            break

        utt = row["utteranceId"]
        mos = float(row["mean"])
        wav_path = SOMOS_AUDIO_ROOT / utt
        if not wav_path.exists():
            # sometimes audio might live elsewhere; make this obvious
            raise FileNotFoundError(f"missing audio file: {wav_path}")

        try:
            dur = float(librosa.get_duration(path=str(wav_path)))
        except Exception:
            y, sr = librosa.load(str(wav_path), sr=None, mono=True)
            dur = float(len(y) / sr)

        picked.append((utt, mos, wav_path, dur))
        total_s += dur

    return pd.DataFrame(picked, columns=["utteranceId", "mos_target", "path", "duration_s"]), total_s, len(df_sys)

def load_audio(path: Path, sr=None):
    y, s = librosa.load(str(path), sr=sr, mono=True)
    return y.astype(np.float32), s

def play_sequence(df_seq: pd.DataFrame, title: str):
    print("\n" + "="*80)
    print(title)
    print("="*80)

    if PLAY_EACH_SEGMENT:
        for i, row in df_seq.reset_index(drop=True).iterrows():
            utt = row["utteranceId"]
            mos_t = row["mos_target"]
            dur = row["duration_s"]
            p = row["path"]

            print(f"\n[{i+1:02d}] {utt} | target MOS={mos_t:.3f} | dur={dur:.2f}s")
            y, sr = load_audio(p, sr=SR)
            display(Audio(y, rate=sr))

    if PLAY_COMBINED_PREVIEW:
        y_all = []
        sr_ref = None

        for row in df_seq.itertuples(index=False):
            y, sr = load_audio(row.path, sr=SR)
            if sr_ref is None:
                sr_ref = sr
            elif sr != sr_ref:
                raise RuntimeError(f"sample rate mismatch: got {sr} vs {sr_ref} for {row.path}")
            y_all.append(y)

        combined = np.concatenate(y_all) if len(y_all) else np.zeros(0, dtype=np.float32)
        total_len_s = float(len(combined) / sr_ref) if sr_ref else 0.0

        preview_len = int(COMBINED_PREVIEW_SECONDS * sr_ref)
        preview = combined[:preview_len]

        print(f"\nCombined preview: {min(COMBINED_PREVIEW_SECONDS, total_len_s):.2f}s of {total_len_s:.2f}s total")
        display(Audio(preview, rate=sr_ref))



# main: load MOS list once, then loop systems
df_all = parse_somos_mos_list(MOS_LIST)
print("loaded:", MOS_LIST, "rows=", len(df_all))

for sid in SYSTEM_IDS:
    df_pick, total_s, n_avail = pick_system_sequence(
        df_all=df_all,
        system_id=sid,
        target_seconds=TARGET_SECONDS,
        start_index=START_INDEX,
        max_files=MAX_FILES
    )

    print("\n" + "-"*80)
    print(f"SYSTEM_ID={sid.zfill(3)} -> available utterances: {n_avail}")
    print(f"picked: {len(df_pick)}")
    print(f"picked duration: {total_s:.2f}s (target={TARGET_SECONDS:.2f}s)")
    display(df_pick[["utteranceId", "mos_target", "duration_s"]].head(50))

    play_sequence(df_pick, f"SOMOS system {sid.zfill(3)} — NORMAL order (picked)")

    df_rev = df_pick.iloc[::-1].reset_index(drop=True)
    play_sequence(df_rev, f"SOMOS system {sid.zfill(3)} — REVERSED order (picked[::-1])")


librosa tests

In [ ]:
import csv
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

from local_sqa.modules.ssl_mos import SAMPLING_RATE
from local_sqa.modules.data_loader import LoadAudio



# CONFIG 
MODEL_DIR = "/net/vol/zigor/checkpoints/24"
CHECKPOINT_NAME = "ckpt_best_SRCC.pth"

BVCC_TEST_LIST = Path("/net/db/BVCC/main/DATA/sets/test_mos_list.txt")
BVCC_WAV_ROOT = Path("/net/db/BVCC/main/DATA/wav")
BVCC_SYSTEM_IDS = [
    "sys47c67",
    "sys78aec",
    "sys83aed",
    "sys91caa",
    "sys753b8",
]

SOMOS_TEST_LIST = Path("/net/db/somos/training_files/split1/clean/test_mos_list.txt")
# SOMOS_TEST_LIST = Path("/net/db/somos/training_files/split1/full/test_mos_list.txt")
SOMOS_WAV_ROOT = Path("/net/db/somos/audios")
SOMOS_SYSTEM_IDS = ["061", "057", "110", "124", "191"]

MODES_TO_RUN = ["forward", "reverse", "random"]

TARGET_SECONDS = 180.0
START_INDEX = 0
MAX_FILES = 5000

RANDOM_SEED = 1234

# librosa trim params (stell hier rum)
TRIM_TOP_DB = 60
TRIM_FRAME_LENGTH = 2048
TRIM_HOP_LENGTH = 512
TRIM_REF = np.max

OUT_DIR = Path.cwd() / "silence_trim_study"
OUT_DIR.mkdir(parents=True, exist_ok=True)



# AUDIO LOADER 
AUDIO_LOADER = LoadAudio(
    audio_path_keys="audio_path.observation",
    target_sampling_rate=SAMPLING_RATE,
    resample=True,
)

def load_wav(path: Path) -> np.ndarray:
    ex = {"audio_path": {"observation": str(path)}}
    ex = AUDIO_LOADER(ex)
    wav = ex["audio"].astype(np.float32)
    if wav.ndim != 1:
        raise ValueError(f"expected 1D audio, got {wav.shape} for {path}")
    return wav



# LIST PARSE
def parse_bvcc_list(list_path: Path):
    items = []
    for ln in list_path.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        fn, mos = ln.split(",", 1)
        items.append((fn.strip(), float(mos)))
    return items

def parse_somos_list(list_path: Path):
    items = []
    for i, ln in enumerate(list_path.read_text().splitlines()):
        ln = ln.strip()
        if not ln:
            continue
        if i == 0 and ln.lower().startswith("utteranceid"):
            continue
        utt, mos = ln.split(",", 1)
        items.append((utt.strip(), float(mos)))
    return items

def extract_system_id_somos(utterance_id: str) -> str:
    u = utterance_id.strip()
    if u.endswith(".wav"):
        u = u[:-4]
    if "_" not in u:
        return "unknown"
    tail = u.split("_")[-1]
    if tail.isdigit():
        return tail.zfill(3)
    return "unknown"


# PACKET BUILD

def order_packet(packet, mode: str, system_id: str):
    if mode == "forward":
        return packet
    if mode == "reverse":
        return list(reversed(packet))
    if mode == "random":
        h = zlib.adler32(system_id.encode("utf-8")) & 0xFFFFFFFF
        seed = (RANDOM_SEED ^ h) & 0xFFFFFFFF
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(packet)).tolist()
        return [packet[i] for i in idx]
    raise ValueError(f"unknown mode: {mode}")

def build_packet_bvcc(items, system_id: str, target_seconds: float):
    filtered = [(fn, mos) for (fn, mos) in items if fn.startswith(f"{system_id}-")]
    if not filtered:
        raise ValueError(f"no items found for SYSTEM_ID={system_id}")
    start = START_INDEX % len(filtered)
    ordered = filtered[start:] + filtered[:start]

    packet = []
    total_s = 0.0
    for fn, mos_t in ordered:
        if total_s >= target_seconds or len(packet) >= MAX_FILES:
            break
        p = BVCC_WAV_ROOT / fn
        wav = load_wav(p)
        dur = len(wav) / SAMPLING_RATE
        packet.append((fn, float(mos_t), p, float(dur)))
        total_s += float(dur)
    return packet, float(total_s), len(filtered)

def build_packet_somos(items, system_id: str, target_seconds: float):
    sid = str(system_id).zfill(3)
    filtered = [(utt, mos) for (utt, mos) in items if extract_system_id_somos(utt) == sid]
    if not filtered:
        sample = [extract_system_id_somos(u) for (u, _) in items[:30]]
        raise ValueError(f"no items found for SYSTEM_ID={sid}. example parsed ids: {sample}")
    start = START_INDEX % len(filtered)
    ordered = filtered[start:] + filtered[:start]

    packet = []
    total_s = 0.0
    for utt, mos_t in ordered:
        if total_s >= target_seconds or len(packet) >= MAX_FILES:
            break
        p = SOMOS_WAV_ROOT / utt
        if not p.exists():
            alt = SOMOS_WAV_ROOT / f"{utt}.wav"
            if alt.exists():
                p = alt
        if not p.exists():
            raise FileNotFoundError(f"missing audio file: {p}")
        wav = load_wav(p)
        dur = len(wav) / SAMPLING_RATE
        packet.append((utt, float(mos_t), p, float(dur)))
        total_s += float(dur)
    return packet, float(total_s), len(filtered)

def save_packet_csv(path: Path, dataset: str, system_id: str, packet, total_s: float, sys_count: int, list_path: Path):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["dataset", dataset])
        w.writerow(["system_id", system_id])
        w.writerow(["list_path", str(list_path)])
        w.writerow(["target_seconds", TARGET_SECONDS])
        w.writerow(["start_index", START_INDEX])
        w.writerow(["sys_count", sys_count])
        w.writerow(["packet_len", len(packet)])
        w.writerow(["packet_total_s", total_s])
        w.writerow([])
        w.writerow(["utt_or_fn", "mos_target", "path", "dur_s"])
        for uid, mos_t, p, dur in packet:
            w.writerow([uid, mos_t, str(p), dur])



def trim_stats(wav: np.ndarray):
    y_trim, idx = librosa.effects.trim(
        wav.astype(np.float32),
        top_db=TRIM_TOP_DB,
        frame_length=TRIM_FRAME_LENGTH,
        hop_length=TRIM_HOP_LENGTH,
        ref=TRIM_REF,
    )
    i0, i1 = int(idx[0]), int(idx[1])
    i0 = max(0, min(i0, len(wav)))
    i1 = max(0, min(i1, len(wav)))
    lead_s = i0 / SAMPLING_RATE
    trail_s = (len(wav) - i1) / SAMPLING_RATE
    dur_s = len(wav) / SAMPLING_RATE
    keep_s = (i1 - i0) / SAMPLING_RATE
    keep_ratio = keep_s / (dur_s + 1e-12)
    return lead_s, trail_s, dur_s, keep_s, keep_ratio


def analyze_dataset(dataset: str):
    if dataset == "bvcc":
        items = parse_bvcc_list(BVCC_TEST_LIST)
        system_ids = BVCC_SYSTEM_IDS
        build_packet = build_packet_bvcc
        list_path = BVCC_TEST_LIST
        tag = "bvcc"
    elif dataset == "somos":
        items = parse_somos_list(SOMOS_TEST_LIST)
        system_ids = [str(x).zfill(3) for x in SOMOS_SYSTEM_IDS]
        build_packet = build_packet_somos
        list_path = SOMOS_TEST_LIST
        tag = SOMOS_TEST_LIST.parent.name  # clean/full
    else:
        raise ValueError("dataset must be 'bvcc' or 'somos'")

    base = OUT_DIR / dataset / tag
    base.mkdir(parents=True, exist_ok=True)

    rows = []
    for sid in system_ids:
        packet, total_s, sys_count = build_packet(items, sid, TARGET_SECONDS)
        pkt_csv = base / f"packet_{dataset}_{tag}_{sid}_t{int(TARGET_SECONDS)}_s{START_INDEX}.csv"
        save_packet_csv(pkt_csv, dataset, sid, packet, total_s, sys_count, list_path)

        for mode in MODES_TO_RUN:
            pkt = order_packet(packet, mode, sid)
            for i, (uid, mos_t, p, dur) in enumerate(pkt, start=1):
                wav = load_wav(p)
                lead_s, trail_s, dur_s, keep_s, keep_ratio = trim_stats(wav)
                rows.append({
                    "dataset": dataset,
                    "tag": tag,
                    "system_id": sid,
                    "mode": mode,
                    "seg_index": i,
                    "utt_or_fn": uid,
                    "path": str(p),
                    "dur_s": dur_s,
                    "lead_s": lead_s,
                    "trail_s": trail_s,
                    "trim_total_s": lead_s + trail_s,
                    "keep_s": keep_s,
                    "keep_ratio": keep_ratio,
                    "trim_top_db": TRIM_TOP_DB,
                    "frame_length": TRIM_FRAME_LENGTH,
                    "hop_length": TRIM_HOP_LENGTH,
                })

    df = pd.DataFrame(rows)
    out_csv = base / "silence_stats.csv"
    df.to_csv(out_csv, index=False)
    print(f"wrote: {out_csv}")

    # quick sanity summary
    thr = 0.02  # 20 ms
    summ = (df.assign(
        start_silence=df["lead_s"] > thr,
        end_silence=df["trail_s"] > thr
    )
    .groupby(["dataset","tag","system_id","mode"])
    .agg(
        n=("lead_s","size"),
        p_start_silence=("start_silence","mean"),
        p_end_silence=("end_silence","mean"),
        lead_ms_med=("lead_s", lambda x: 1000*np.median(x)),
        trail_ms_med=("trail_s", lambda x: 1000*np.median(x)),
        trim_ms_med=("trim_total_s", lambda x: 1000*np.median(x)),
    )
    .reset_index())

    return df, summ



# RUN 
df_bvcc, summ_bvcc = analyze_dataset("bvcc")
df_somos, summ_somos = analyze_dataset("somos")

display(summ_bvcc)
display(summ_somos)


In [ ]:
from pathlib import Path
import numpy as np
import librosa
from IPython.display import Audio, display

from local_sqa.modules.ssl_mos import SAMPLING_RATE
from local_sqa.modules.data_loader import LoadAudio


AUDIO_PATH = Path("/net/db/BVCC/main/DATA/wav/sys91caa-...")  


TRIM_TOP_DB = 40
TRIM_FRAME_LENGTH = 512
TRIM_HOP_LENGTH = 64
TRIM_REF = np.max


AUDIO_LOADER = LoadAudio(
    audio_path_keys="audio_path.observation",
    target_sampling_rate=SAMPLING_RATE,
    resample=True,
)

def load_wav(path: Path) -> np.ndarray:
    ex = {"audio_path": {"observation": str(path)}}
    ex = AUDIO_LOADER(ex)
    wav = ex["audio"].astype(np.float32)
    if wav.ndim != 1:
        raise ValueError(f"expected 1D audio, got {wav.shape} for {path}")
    return wav

wav = load_wav(AUDIO_PATH)
y_trim, (i0, i1) = librosa.effects.trim(
    wav,
    top_db=TRIM_TOP_DB,
    frame_length=TRIM_FRAME_LENGTH,
    hop_length=TRIM_HOP_LENGTH,
    ref=TRIM_REF,
)

lead_s = i0 / SAMPLING_RATE
trail_s = (len(wav) - i1) / SAMPLING_RATE
dur_s = len(wav) / SAMPLING_RATE
keep_s = len(y_trim) / SAMPLING_RATE

print(f"path: {AUDIO_PATH}")
print(f"dur_s: {dur_s:.3f}")
print(f"trim start idx: {i0} -> lead_s: {lead_s*1000:.1f} ms")
print(f"trim end   idx: {i1} -> trail_s: {trail_s*1000:.1f} ms")
print(f"kept: {keep_s:.3f} s  (removed total: {(lead_s+trail_s):.3f} s)")
print(f"params: top_db={TRIM_TOP_DB}, frame={TRIM_FRAME_LENGTH}, hop={TRIM_HOP_LENGTH}")

print("\n--- original ---")
display(Audio(wav, rate=SAMPLING_RATE))

print("\n--- trimmed ---")
display(Audio(y_trim, rate=SAMPLING_RATE))


SILENCE TESTS BVCC

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display


# config
BVCC_TEST_LIST = Path("/net/db/BVCC/main/DATA/sets/test_mos_list.txt")
BVCC_WAV_ROOT  = Path("/net/db/BVCC/main/DATA/wav")

SYSTEM_ID = "sys6c11c"          
TARGET_SECONDS = 180.0
START_INDEX = 0
MAX_FILES = 5000

ORDER = "forward"              # "forward", "reverse", "random"
RANDOM_SEED = 1234


SR_TARGET = 16000              # None keeps original sr; keep fixed for safe concat
MONO = True

# vad / silence handling (librosa-based)
TOP_DB = 35
FRAME_LENGTH = 2048
HOP_LENGTH = 512
PAD_MS = 40

# artificial pauses
INSERT_PAUSE_SECONDS = 5.0     

# visualization
MAX_FILES_TO_VISUALIZE = 10
PLAY_ORIG = True
PLAY_CUT_ONLY = True
PLAY_WITH_PAUSES = True
PLOT_AFTER_AUDIO = True

# combined preview
PLAY_COMBINED_PREVIEW = True
COMBINED_PREVIEW_SECONDS = 20.0



# helpers
def parse_bvcc_list(list_path: Path):
    items = []
    for ln in list_path.read_text().splitlines():
        ln = ln.strip()
        if not ln:
            continue
        fn, mos = ln.split(",")
        items.append((fn.strip(), float(mos)))
    return items


def load_audio(path: Path, sr_target=SR_TARGET, mono=MONO):
    y, sr = librosa.load(str(path), sr=sr_target, mono=mono)
    return y.astype(np.float32), int(sr)


def pick_system_sequence(items, system_id: str, target_seconds: float, start_index: int = 0, max_files: int = 5000):
    filtered = [(fn, mos) for (fn, mos) in items if fn.startswith(f"{system_id}-")]
    if not filtered:
        raise ValueError(f"no items found for SYSTEM_ID={system_id}")

    start = start_index % len(filtered)
    ordered = filtered[start:] + filtered[:start]

    seq = []
    total_s = 0.0

    for fn, mos_t in ordered:
        if total_s >= target_seconds or len(seq) >= max_files:
            break

        p = BVCC_WAV_ROOT / fn
        try:
            dur = float(librosa.get_duration(path=str(p)))
        except Exception:
            y, sr = load_audio(p, sr_target=None, mono=True)
            dur = float(len(y) / sr)

        seq.append((fn, float(mos_t), p, float(dur)))
        total_s += float(dur)

    return seq, float(total_s), len(filtered)


def order_df(df: pd.DataFrame, order: str, seed: int):
    if order == "forward":
        return df.reset_index(drop=True)
    if order == "reverse":
        return df.iloc[::-1].reset_index(drop=True)
    if order == "random":
        rng = np.random.default_rng(seed)
        idx = rng.permutation(len(df))
        return df.iloc[idx].reset_index(drop=True)
    raise ValueError(f"unknown ORDER: {order}")


def vad_intervals_librosa(y: np.ndarray, sr: int):
    intervals = librosa.effects.split(
        y,
        top_db=TOP_DB,
        frame_length=FRAME_LENGTH,
        hop_length=HOP_LENGTH
    )
    if len(intervals) == 0:
        return []

    pad = int((PAD_MS / 1000.0) * sr)
    out = []
    for s, e in intervals:
        s2 = max(0, int(s) - pad)
        e2 = min(len(y), int(e) + pad)
        if e2 > s2:
            out.append((s2, e2))

    out.sort()
    merged = [out[0]]
    for s, e in out[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def build_processed(y: np.ndarray, sr: int, intervals, pause_s: float):
    if not intervals:
        return np.zeros(0, dtype=np.float32), []

    pause_n = int(round(pause_s * sr))
    pause = np.zeros(pause_n, dtype=np.float32)

    chunks = []
    proc_map = []
    proc_cursor = 0

    for k, (s, e) in enumerate(intervals):
        seg = y[s:e]
        chunks.append(seg)

        ps = proc_cursor
        pe = proc_cursor + len(seg)
        proc_map.append({"type": "speech", "orig_s": s, "orig_e": e, "proc_s": ps, "proc_e": pe})
        proc_cursor = pe

        if k < len(intervals) - 1 and pause_n > 0:
            chunks.append(pause)
            ps2 = proc_cursor
            pe2 = proc_cursor + pause_n
            proc_map.append({"type": "pause", "orig_s": None, "orig_e": None, "proc_s": ps2, "proc_e": pe2})
            proc_cursor = pe2

    y_proc = np.concatenate(chunks).astype(np.float32) if chunks else np.zeros(0, dtype=np.float32)
    return y_proc, proc_map


def plot_waveforms(y, sr, intervals, y_cut, cut_map, y_pause, pause_map, title):
    t = np.arange(len(y)) / sr
    tc = np.arange(len(y_cut)) / sr if len(y_cut) else np.zeros(0)
    tp = np.arange(len(y_pause)) / sr if len(y_pause) else np.zeros(0)

    plt.figure(figsize=(16, 9))

    ax1 = plt.subplot(3, 1, 1)
    ax1.plot(t, y, linewidth=0.8)
    ax1.set_title(f"{title} — original (green spans = kept speech)")
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("amp")
    for s, e in intervals:
        ax1.axvspan(s / sr, e / sr, alpha=0.25)

    ax2 = plt.subplot(3, 1, 2)
    if len(y_cut):
        ax2.plot(tc, y_cut, linewidth=0.8)
    ax2.set_title(f"{title} — CUT ONLY (all silence removed, 0s artificial pauses)")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("amp")

    ax3 = plt.subplot(3, 1, 3)
    if len(y_pause):
        ax3.plot(tp, y_pause, linewidth=0.8)
    ax3.set_title(f"{title} — CUT + {INSERT_PAUSE_SECONDS}s pauses between chunks (orange spans = pauses)")
    ax3.set_xlabel("time (s)")
    ax3.set_ylabel("amp")
    for m in pause_map:
        if m["type"] == "pause":
            ax3.axvspan(m["proc_s"] / sr, m["proc_e"] / sr, alpha=0.25)

    plt.tight_layout()
    plt.show()


# main: pick files
items = parse_bvcc_list(BVCC_TEST_LIST)
picked, total_s, sys_count = pick_system_sequence(
    items,
    system_id=SYSTEM_ID,
    target_seconds=TARGET_SECONDS,
    start_index=START_INDEX,
    max_files=MAX_FILES
)

df = pd.DataFrame(picked, columns=["filename", "mos_target", "path", "duration_s"])
df = order_df(df, ORDER, RANDOM_SEED)

print(f"SYSTEM_ID={SYSTEM_ID} | available in test list: {sys_count}")
print(f"picked files: {len(df)} | picked duration: {df['duration_s'].sum():.2f}s (target={TARGET_SECONDS:.2f}s)")
display(df[["filename", "mos_target", "duration_s"]].head(50))



# per-file: play orig -> play cut-only -> play with pauses -> plot
n_vis = min(MAX_FILES_TO_VISUALIZE, len(df))
processed_rows = []

for i, row in df.iloc[:n_vis].reset_index(drop=True).iterrows():
    p = Path(row["path"])
    y, sr = load_audio(p, sr_target=SR_TARGET, mono=MONO)

    intervals = vad_intervals_librosa(y, sr)

    # 1) cut-only (0s pauses)
    y_cut, cut_map = build_processed(y, sr, intervals, pause_s=0.0)

    # 2) cut + pauses (5s)
    y_pause, pause_map = build_processed(y, sr, intervals, pause_s=INSERT_PAUSE_SECONDS)

    orig_s = len(y) / sr
    cut_s = len(y_cut) / sr if len(y_cut) else 0.0
    pause_s = len(y_pause) / sr if len(y_pause) else 0.0
    speech_s = sum((e - s) for s, e in intervals) / sr if intervals else 0.0
    removed_s = max(orig_s - speech_s, 0.0)

    title = f"[{i+1:02d}] {row['filename']} | targetMOS={row['mos_target']:.3f}"
    print(f"\n{title}")
    print(f"orig={orig_s:.2f}s | speech≈{speech_s:.2f}s | removed_silence≈{removed_s:.2f}s | cut_only={cut_s:.2f}s | cut+pauses={pause_s:.2f}s | chunks={len(intervals)} | sr={sr}")

    # audio BEFORE plots
    if PLAY_ORIG:
        print("audio: original")
        display(Audio(y, rate=sr))

    if PLAY_CUT_ONLY:
        print("audio: cut-only (silence removed, 0s artificial pauses)")
        display(Audio(y_cut, rate=sr))

    if PLAY_WITH_PAUSES:
        print(f"audio: cut + {INSERT_PAUSE_SECONDS}s pauses between chunks")
        display(Audio(y_pause, rate=sr))

    if PLOT_AFTER_AUDIO:
        plot_waveforms(y, sr, intervals, y_cut, cut_map, y_pause, pause_map, title)

    processed_rows.append({
        "filename": row["filename"],
        "mos_target": row["mos_target"],
        "orig_s": orig_s,
        "speech_s": speech_s,
        "removed_silence_s": removed_s,
        "cut_only_s": cut_s,
        "cut_plus_pauses_s": pause_s,
        "chunks": len(intervals),
        "path": str(p),
    })

df_proc = pd.DataFrame(processed_rows)
display(df_proc)



# combined preview (original order) for quick sanity
if PLAY_COMBINED_PREVIEW and len(df) > 0:
    y_all = []
    y_all_cut = []
    y_all_pause = []
    sr_ref = None

    for row in df.itertuples(index=False):
        y, sr = load_audio(Path(row.path), sr_target=SR_TARGET, mono=MONO)
        sr_ref = sr if sr_ref is None else sr_ref
        if sr != sr_ref:
            raise RuntimeError(f"sample rate mismatch: {sr} vs {sr_ref} for {row.path}")

        intervals = vad_intervals_librosa(y, sr_ref)
        y_cut, _ = build_processed(y, sr_ref, intervals, pause_s=0.0)
        y_pause, _ = build_processed(y, sr_ref, intervals, pause_s=INSERT_PAUSE_SECONDS)

        y_all.append(y)
        y_all_cut.append(y_cut)
        y_all_pause.append(y_pause)

    comb = np.concatenate(y_all).astype(np.float32) if y_all else np.zeros(0, dtype=np.float32)
    comb_cut = np.concatenate(y_all_cut).astype(np.float32) if y_all_cut else np.zeros(0, dtype=np.float32)
    comb_pause = np.concatenate(y_all_pause).astype(np.float32) if y_all_pause else np.zeros(0, dtype=np.float32)

    n_prev = int(COMBINED_PREVIEW_SECONDS * sr_ref)
    prev = comb[:n_prev]
    prev_cut = comb_cut[:n_prev]
    prev_pause = comb_pause[:n_prev]

    print("\n" + "=" * 80)
    print(f"combined preview (first {COMBINED_PREVIEW_SECONDS}s) | SYSTEM_ID={SYSTEM_ID} | ORDER={ORDER}")
    print("=" * 80)

    print("audio: combined original preview")
    display(Audio(prev, rate=sr_ref))

    print("audio: combined cut-only preview (0s artificial pauses)")
    display(Audio(prev_cut, rate=sr_ref))

    print(f"audio: combined cut+pauses preview ({INSERT_PAUSE_SECONDS}s pauses)")
    display(Audio(prev_pause, rate=sr_ref))
